# Full SFT + evaluation notebook (Vast.ai, single GPU) -- A10

Runs SFT (which hadn't actually completed yet -- the earlier console-log capture only covered
pretrain) and the *full* (unsampled) benchmarks against the resulting `a10` SFT checkpoint,
same method as `kaggle/kaggle_eval.ipynb` used for `d4`/`d6`:

- `scripts/chat_sft.py`: 1 SmolTalk epoch, same config as `d4`/`d6`'s SFT.
- `scripts/chat_eval.py`: ARC-Easy, ARC-Challenge, MMLU, GSM8K, HumanEval.
- `scripts/eval_blimp.py`: BLiMP (Warstadt et al. 2020), 67 categories x 1000 pairs.

Run this in the browser Jupyter app on the same instance `vastai/run_a10.sh` trained on (Instance
Portal -> Jupyter) -- reuses the `~/repo` clone, `~/nanochat_cache`, and `~/.config/rclone/rclone.conf`
already on that box if present, only falling back to a fresh `git clone`/rclone pull/credential
prompt if something's missing (e.g. a fresh instance). Download this notebook when done
(File -> Download) and archive it under `vastai/runs/`, same convention as `kaggle/runs/`.

No credentials are stored in this file -- Cell 2 reuses the existing `rclone.conf` from
`run_a10.sh` on this same box, and only prompts interactively (`getpass`) if that's missing too.
Never hardcode real credentials into this notebook, even "temporarily" -- this repo is public,
and a committed secret stays recoverable from git history even after being removed in a later
commit.

## Cell 0: clean up build caches before doing anything else

`run_a10.sh` already hit a disk-full crash once (16GB filled by accumulated pretrain
checkpoints -- see RESEARCH_LOG.md 2026-08-11). Clear leftover build caches from that run
first. (Not touching `base_checkpoints/a10` here -- SFT in Cell 3 needs it.)

In [ ]:
import os
import subprocess

print("Before cleanup:")
!df -h /

# Build caches from run_a10.sh -- safe to drop, nothing here is needed again.
!rm -rf ~/.cache/uv ~/.cache/pip
!rm -rf ~/.cargo/registry/cache ~/.cargo/registry/src
subprocess.run(["bash", "-lc", "apt-get clean 2>/dev/null || true"])

# NOT touching base_checkpoints/a10 here -- SFT (Cell 3 below) trains from it, so it's still
# needed. Only safe to delete *after* SFT has run and synced (see the optional cell at the end).

print("After cleanup:")
!df -h /

## Cell 1: repo + deps (reuses `~/repo` if `run_a10.sh` already set it up)

In [ ]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/nadeko0/nanochat-ru.git"
REPO_DIR = os.path.expanduser("~/repo")

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print("Repo already present, pulling latest...")
    !git -C {REPO_DIR} pull
else:
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)

def have(cmd):
    return subprocess.run(["bash", "-lc", f"command -v {cmd}"], capture_output=True).returncode == 0

if not have("uv"):
    !curl -LsSf https://astral.sh/uv/install.sh | sh
os.environ["PATH"] = f"{os.path.expanduser('~/.local/bin')}:{os.environ['PATH']}"

if not have("cargo"):
    !curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y
os.environ["PATH"] = f"{os.path.expanduser('~/.cargo/bin')}:{os.environ['PATH']}"

if not have("rclone"):
    !curl https://rclone.org/install.sh | sudo bash

!uv pip install --system --python {sys.executable} --extra gpu -r pyproject.toml

print("Cell 1 done.")

## Cell 2: rclone config (reuses `~/.config/rclone/rclone.conf` from `run_a10.sh` if present) + pull tokenizer/base checkpoint

In [ ]:
import os

NANOCHAT_BASE_DIR = os.path.expanduser("~/nanochat_cache")
os.environ["NANOCHAT_BASE_DIR"] = NANOCHAT_BASE_DIR

rclone_conf_path = os.path.expanduser("~/.config/rclone/rclone.conf")
if os.path.exists(rclone_conf_path):
    # run_a10.sh already wrote this on this same box -- same instance, same disk, reuse it.
    # No need to re-enter credentials (and nothing here ever gets committed either way).
    print(f"Found existing {rclone_conf_path} from run_a10.sh -- reusing it, no prompt needed.")
else:
    print("No rclone.conf on this box yet. Enter the same 4 credentials used for run_a10.sh:")
    from getpass import getpass
    client_id = getpass("GDRIVE_CLIENT_ID: ")
    client_secret = getpass("GDRIVE_CLIENT_SECRET: ")
    oauth_token = getpass("GDRIVE_OAUTH_TOKEN (the whole JSON blob): ")
    folder_id = getpass("GDRIVE_FOLDER_ID: ")

    os.makedirs(os.path.dirname(rclone_conf_path), exist_ok=True)
    with open(rclone_conf_path, "w") as f:
        f.write(
            "[gdrive]\n"
            "type = drive\n"
            "scope = drive\n"
            f"client_id = {client_id}\n"
            f"client_secret = {client_secret}\n"
            f"token = {oauth_token}\n"
            f"root_folder_id = {folder_id}\n"
            "team_drive =\n"
        )
    del client_id, client_secret, oauth_token, folder_id

# Tokenizer -- needed either way.
tokenizer_ok = os.path.exists(os.path.join(NANOCHAT_BASE_DIR, "tokenizer", "tokenizer.pkl"))
if not tokenizer_ok:
    !rclone copy gdrive:tokenizer {NANOCHAT_BASE_DIR}/tokenizer --checksum -v

# Base (pretrain) checkpoint -- SFT (Cell 3) trains from this, and only from the LAST step
# (1905). Drive has every --save-every step from the whole pretrain run (100, 200, ..., 1905) --
# a plain `rclone copy` of the whole remote dir pulls all of them and re-triggers the same
# disk-full crash pretrain already had (hit this exact bug once already). --include restricts
# the pull to just the one step SFT actually needs.
base_ckpt_dir = os.path.join(NANOCHAT_BASE_DIR, "base_checkpoints", "a10")
LAST_STEP = 1905
step_str = f"{LAST_STEP:06d}"
base_ok = os.path.exists(os.path.join(base_ckpt_dir, f"model_{step_str}.pt")) and \
          os.path.exists(os.path.join(base_ckpt_dir, f"optim_{step_str}_rank0.pt"))
if not base_ok:
    print(f"Base checkpoint step {LAST_STEP} not fully local, pulling just that step from Drive...")
    !rclone copy gdrive:base_checkpoints/a10 {base_ckpt_dir} --include "*{step_str}*" --checksum -v

print("Base checkpoint ready:")
!ls {base_ckpt_dir}

## Cell 3: run SFT for a10 (this hadn't actually completed yet -- the recovered console log only covered pretrain)

In [ ]:
import os
os.chdir(os.path.expanduser("~/repo"))

# Same config run_a10.sh's Step 5 used (1 SmolTalk epoch, same as d4/d6's SFT).
!python3 -m scripts.chat_sft \
    --model-tag=a10 --mmlu-epochs=0 --gsm8k-epochs=0 \
    --num-iterations=500 --chatcore-every=-1 --eval-every=100 --run=dummy

# Quick sanity check + sync to Drive right after, same as run_a10.sh did.
!python3 kaggle/sync_checkpoints.py --remote gdrive: --once --log-file ~/sync.log
!python3 -m scripts.chat_cli -i sft -g a10 -p "hi"
!python3 -m scripts.chat_cli -i sft -g a10 -p "What is your name?"

## Cell 4: full chat_eval.py -- a10

In [ ]:
import os
os.chdir(os.path.expanduser("~/repo"))

# Full run, no -x limit. If this is taking too long, interrupt and rerun with e.g. `-x 200`
# (chat_eval.py's --max-problems flag) for a faster, still-informative sample.
!python3 -m scripts.chat_eval -i sft -g a10 2>&1 | tee ~/chat_eval_a10.log

## Cell 5: full BLiMP eval -- a10

In [ ]:
import os
os.chdir(os.path.expanduser("~/repo"))

# All 67 categories, 1000 pairs each, batched. If too slow, lower --max-pairs (e.g. 200).
!python3 -m scripts.eval_blimp -i sft -g a10 --batch-size 64 2>&1 | tee ~/blimp_a10.log